# Storing Common Information ke MongoDB Atlas

Notebook ini bertugas untuk:
1. Load dataset Common Information dari file JSON
2. Membuat Haystack `Document` untuk setiap FAQ entry
3. Membuat storing pipeline (Embedder → DocumentWriter)
4. Menyimpan semua dokumen + embedding ke MongoDB Atlas collection `common_information`

## Strategi Penyimpanan
- **Database:** `depato_store` (sama dengan produk)
- **Collection:** `common_information` (koleksi baru, terpisah dari produk)
- **Dokumen:** Setiap FAQ entry dijadikan satu `Document` Haystack
  - `content` = pertanyaan + jawaban (untuk embedding kontekstual)
  - `meta.topic` = kategori topik (shipping, refund, dll.)
  - `meta.question` = pertanyaan asli
- **Embedding:** `SentenceTransformers` (all-mpnet-base-v2, 768 dimensi) — sama dengan produk agar konsisten
- **Duplicate Policy:** OVERWRITE — aman untuk re-run notebook

## Setup Environment Variables

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()  # Load dari .env file

# Pastikan sudah ada di .env
# MONGO_CONNECTION_STRING=mongodb+srv://...
print("MONGO_CONNECTION_STRING:", os.environ.get("MONGO_CONNECTION_STRING", "NOT SET")[:30], "...")

MONGO_CONNECTION_STRING: mongodb+srv://alfiahzalfa:zalf ...


## Load Dataset Common Information

In [10]:
import json

with open('../data/common_information.json', 'r', encoding='utf-8') as f:
    common_info_data = json.load(f)

print(f"Total dokumen: {len(common_info_data)}")
print("\nContoh dokumen pertama:")
print(json.dumps(common_info_data[0], indent=2))

Total dokumen: 29

Contoh dokumen pertama:
{
  "topic": "shipping",
  "question": "How long does shipping take?",
  "answer": "Shipping time depends on the destination and the selected shipping method. Standard shipping typically takes 5-7 business days within the country. Expedited shipping takes 2-3 business days. For international orders, please allow 10-20 business days. You will receive a tracking number by email once your order has been dispatched."
}


## Konversi ke Haystack Documents

In [11]:
from haystack import Document

documents = []
for item in common_info_data:
    # Content = gabungan question + answer untuk embedding yang lebih kontekstual
    content = f"Question: {item['question']}\nAnswer: {item['answer']}"
    
    doc = Document(
        content=content,
        meta={
            "topic": item["topic"],
            "question": item["question"],
        }
    )
    documents.append(doc)

print(f"Total Haystack Documents: {len(documents)}")
print("\nContoh Document pertama:")
print(documents[0])

Total Haystack Documents: 29

Contoh Document pertama:
Document(id=b17d6dcc43e48b96f17bb8e060c8193f60f3e4c04e7e29c7e1e7d189e73e357c, content: 'Question: How long does shipping take?
Answer: Shipping time depends on the destination and the sele...', meta: {'topic': 'shipping', 'question': 'How long does shipping take?'})


## Setup MongoDB Atlas — Pastikan Collection Ada

In [12]:
from pymongo import MongoClient

client = MongoClient(os.environ['MONGO_CONNECTION_STRING'])
db = client['depato_store']

# Buat collection common_information jika belum ada
if 'common_information' not in db.list_collection_names():
    db.create_collection('common_information')
    print('Collection common_information berhasil dibuat.')
else:
    print('Collection common_information sudah ada.')

print(f"\nCollections di depato_store: {db.list_collection_names()}")

Collection common_information sudah ada.

Collections di depato_store: ['categories', 'common_information', 'materials', 'products']


## Buat MongoDBAtlas Document Store untuk Common Information

In [13]:
from haystack_integrations.document_stores.mongodb_atlas import MongoDBAtlasDocumentStore

# Document store khusus untuk common information
common_info_store = MongoDBAtlasDocumentStore(
    database_name="depato_store",
    collection_name="common_information",      # Collection terpisah dari produk
    vector_search_index="common_info_vector_index",  # Index vector baru
    full_text_search_index="common_info_search_index",
)

print("Document store berhasil dibuat.")
print(f"Collection: {common_info_store.collection_name}")

Document store berhasil dibuat.
Collection: common_information


## Membuat Storing Pipeline

Pipeline:
```
documents
    │
    ▼
SentenceTransformersDocumentEmbedder  ← Generate embedding 768 dimensi
    │
    ▼
DocumentWriter (OVERWRITE policy)     ← Simpan ke MongoDB Atlas
```

In [14]:
from haystack import Pipeline
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.writers import DocumentWriter
from haystack.document_stores.types import DuplicatePolicy

storing_pipeline = Pipeline()

storing_pipeline.add_component(
    "embedder",
    SentenceTransformersDocumentEmbedder()  # all-mpnet-base-v2, 768 dimensi
)
storing_pipeline.add_component(
    "writer",
    DocumentWriter(
        document_store=common_info_store,
        policy=DuplicatePolicy.OVERWRITE  # Aman untuk re-run
    )
)

storing_pipeline.connect("embedder", "writer")

print(storing_pipeline)

🚅 Components
  - embedder: SentenceTransformersDocumentEmbedder
  - writer: DocumentWriter
🛤️ Connections
  - embedder.documents -> writer.documents (List[Document])



## Jalankan Pipeline — Simpan ke MongoDB Atlas

In [15]:
result = storing_pipeline.run({
    "embedder": {
        "documents": documents
    }
})

print(f"✅ Berhasil menyimpan {result['writer']['documents_written']} dokumen ke MongoDB Atlas")
print(f"   Collection: common_information")
print(f"   Database: depato_store")

Batches: 100%|██████████| 1/1 [00:02<00:00,  2.73s/it]


✅ Berhasil menyimpan 29 dokumen ke MongoDB Atlas
   Collection: common_information
   Database: depato_store


## Verifikasi — Cek Data yang Tersimpan

In [16]:
# Cek jumlah dokumen di collection
collection = db['common_information']
count = collection.count_documents({})
print(f"Total dokumen di collection common_information: {count}")

# Tampilkan sample dokumen 
sample = collection.find_one({}, {"_id": 0, "content": 1, "meta": 1, "embedding": 1})
print("\nContoh dokumen tersimpan:")
print(f"  Topic: {sample.get('meta', {}).get('topic')}")
print(f"  Question: {sample.get('meta', {}).get('question')}")
print(f"  Content: {sample.get('content', '')[:100]}...")
print(f"  Embedding dimensions: {len(sample.get('embedding', []))}")


Total dokumen di collection common_information: 29

Contoh dokumen tersimpan:
  Topic: shipping
  Question: How long does shipping take?
  Content: Question: How long does shipping take?
Answer: Shipping time depends on the destination and the sele...
  Embedding dimensions: 768


## Vector Search Index di MongoDB Atlas

Setelah menjalankan notebook ini, kamu perlu membuat **Vector Search Index** di MongoDB Atlas secara manual:

1. Login ke [cloud.mongodb.com](https://cloud.mongodb.com)
2. Masuk ke cluster → **Atlas Search** → **Create Search Index**
3. Pilih **Atlas Vector Search** → **JSON Editor**
4. Pilih collection **`common_information`**
5. Beri nama index: **`common_info_vector_index`**
6. Gunakan konfigurasi berikut:

```json
{
  "mappings": {
    "dynamic": true,
    "fields": {
      "embedding": {
        "type": "knnVector",
        "dimensions": 768,
        "similarity": "cosine"
      }
    }
  }
}
```

Tunggu hingga status index menjadi **Active** sebelum menjalankan aplikasi.